DIVIDED DIFFERENCE 



In [ ]:

def divided_differences(nodes, values):
    ### input: list of nodes [x0, x1, ..., xn] length n+1 & list of values [f(x0), f(x1), ..., f(xn)] length n+1
    ### Output: list of divided differences [f[x0], f[x0, x1], ..., f[x0, x1, ..., xn]] length n+1
    dd = values.copy()

    result = [dd[0]]

    for k in range(1, len(nodes)):
        for i in range(len(nodes) - k):
            dd[i] = (dd[i + 1] - dd[i]) / (nodes[i + k] - nodes[i])

        result.append(dd[0])

    return result

    

LAGRANGE INTERPOLATION 

In [ ]:
def lagrange_interpolation(x_values: list, y_values: list, x: int):
    """
    Computes the Lagrange interpolation polynomial at the point x.

    x_values: list of interpolation nodes, e.g. [0, 1, 2]
    y_values: list of function values, e.g. [1, 3, 2]
    x: the point where we want to evaluate the polynomial
    """
    n = len(x_values)
    result = 0

    for i in range(n):
        # Start computing L_i(x)
        L_i = 1

        for j in range(n):
            if j != i:
                L_i = L_i * (x - x_values[j]) / (x_values[i] - x_values[j])

        # Add y_i * L_i(x) to the result
        result = result + y_values[i] * L_i

    return result

NEWTON INTERPOLATION 

In [ ]:
def newton_coefficients(x_values, y_values):
    """
    Computes the Newton divided difference coefficients.

    x_values: interpolation nodes
    y_values: function values

    returns: list of Newton coefficients
    """

    n = len(x_values)

    # Start with a copy of y_values
    coefficients = y_values.copy()

    # Compute divided differences
    for k in range(1, n):
        for i in range(n - 1, k - 1, -1):
            numerator = coefficients[i] - coefficients[i - 1]
            denominator = x_values[i] - x_values[i - k]
            coefficients[i] = numerator / denominator

    return coefficients

def newton_interpolation(x_values, y_values, x):
    """
    Evaluates the Newton interpolation polynomial at x.
    """

    coefficients = newton_coefficients(x_values, y_values)

    n = len(x_values)

    # Start with the highest coefficient
    result = coefficients[n - 1]

    # Nested multiplication
    for i in range(n - 2, -1, -1):
        result = result * (x - x_values[i]) + coefficients[i]

    return result

NEVILLE INTERPOLATION (extention of newton)

In [ ]:
def neville_interpolation(x_values, y_values, x):
    """
    Evaluates the interpolation polynomial at x using Neville's algorithm.
    """

    n = len(x_values)

    # Start with the y-values
    p = y_values.copy()

    # Build Neville table in-place
    for k in range(1, n):
        for i in range(n - k):
            left = (x_values[i + k] - x) * p[i]
            right = (x - x_values[i]) * p[i + 1]
            denominator = x_values[i + k] - x_values[i]

            p[i] = (left + right) / denominator

    return p[0]

ERROR CONVERGENCE RATE/SPEED (Lagrange, Newton, Nevilles)

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
a = 0
b = 1

n_list = [2, 4, 8, 12, 16, 20]
x_eval = np.linspace(a, b, 1000)
y_true_values = f(x_eval) #the function with x 

errors_list = []

plt.figure(figsize=(8, 6))

for n in n_list:
    x_grid = np.linspace(a, b, n + 1)
    y_grid = f(x_grid)

    lagrange_values = LagrangeInterpolation(x_grid, y_grid, x_eval) #SAME FOR NEWTON AND NEVILLE 

    error = np.max(np.abs(y_true_values - lagrange_values))
    errors_list.append(error)

    plt.plot(x_eval, lagrange_values, label=f"n={n}")

plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Lagrange interpolation of $f(x)=x^{1/4}$ on $[0,1]$")
plt.legend()
plt.show()


errors_array = np.array(errors_list)
n_array = np.array(n_list)

plt.figure(figsize=(8, 6))
plt.loglog(n_array, errors_array, "o-", label="Interpolation error")
plt.xlabel("n")
plt.ylabel("Maximum error")
plt.title("Log-log plot of Lagrange interpolation error")
plt.legend()
plt.show()

ALGEBRAIC CONVERGENCE (loglog plot)

In [ ]:
errors_array = np.array(errors_list)
n_array = np.array(n_list)
# We assume error ~ C * n^(-p)

# FORMULA OF THE ERROR ESTIMATE = log(e_old / e_new) / log(n_new / n_old)
p_values = []

for k in range(len(n_array) - 1):
    e_old = errors_array[k]
    e_new = errors_array[k + 1]

    n_old = n_array[k]
    n_new = n_array[k + 1]

    p_k = np.log(e_old / e_new) / np.log(n_new / n_old)
    p_values.append(p_k)

p_values = np.array(p_values)

# Use the last observed convergence rate as estimate
p_estimated = p_values[-1]

reference = errors_array[-1] * (n_array / n_array[-1])**(-p_estimated)

print("Observed convergence rates:")
for k in range(len(p_values)):
    print(f"from n={n_array[k]} to n={n_array[k+1]}: p ≈ {p_values[k]:.4f}")

print(f"Estimated convergence speed: algebraic with p = {p_estimated:.4f}")

plt.figure(figsize=(8, 6))
plt.loglog(n_array, errors_array, "o-", label="Interpolation error")
plt.loglog(n_array, reference, "--", label=fr"Reference slope $n^{{-{p_estimated:.2f}}}$")
plt.xlabel("n")
plt.ylabel("Maximum error")
plt.title("Log-log plot of interpolation error")
plt.legend()
plt.show()

EXPONENTIAL ERROR CONVERGENCE  (semilog plot)

In [ ]:
errors_array = np.array(errors_list)
n_array = np.array(n_list)

# Manual exponential convergence estimate
# FORMULA USED =  C * exp(-alpha * n)

alpha_values = []

for k in range(len(n_array) - 1):
    e_old = errors_array[k]
    e_new = errors_array[k + 1]

    n_old = n_array[k]
    n_new = n_array[k + 1]

    alpha_k = np.log(e_old / e_new) / (n_new - n_old)
    alpha_values.append(alpha_k)

alpha_values = np.array(alpha_values)

# Use the last observed rate as estimate
alpha_estimated = alpha_values[-1]

# Equivalent rho value: error ~ C * rho^(-n)
rho_estimated = np.exp(alpha_estimated)

reference = errors_array[-1] * np.exp(-alpha_estimated * (n_array - n_array[-1]))

print("Observed exponential convergence rates:")
for k in range(len(alpha_values)):
    print(
        f"from n={n_array[k]} to n={n_array[k+1]}: "
        f"alpha ≈ {alpha_values[k]:.4f}, rho ≈ {np.exp(alpha_values[k]):.4f}"
    )

print(f"Estimated exponential convergence speed: alpha = {alpha_estimated:.4f}")
print(f"Equivalent form: error ~ C * rho^(-n), with rho = {rho_estimated:.4f}")
plt.figure(figsize=(8, 6))
plt.semilogy(n_array, errors_array, "o-", label="Interpolation error")
plt.semilogy(n_array, reference, "--", label=fr"Reference $e^{{-{alpha_estimated:.2f}n}}$")
plt.xlabel("n")
plt.ylabel("Maximum error")
plt.title("Semilog plot of interpolation error")
plt.legend()
plt.show()

SPLINES INTERPOLATION  

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
a = 0 # OUR INTERVAL 
b = 1 

x_eval = np.linspace(a, b, 10001)
y_true_values = x_eval**(1/4) # OUR FUNCTION 

def LinearSplines(set_ordered_gridpoints, values_at_gridpoints, set_evaluationpoints):
    x_grid = set_ordered_gridpoints
    y_grid = values_at_gridpoints
    x_eval = set_evaluationpoints
    splinevalues = []

#finding the interval where x lies 
    for x in x_eval:
        if x <= x_grid[0]:
            i = 0
        elif x >= x_grid[-1]:
            i = len(x_grid) - 2
        else:
            i = 0
            for j in range(len(x_grid) - 1):
                if x_grid[j] <= x <= x_grid[j + 1]:
                    i = j
                    break
#setting left and right x, y from the point for the formula 
        x_left, x_right = x_grid[i], x_grid[i+1]
        y_left, y_right = y_grid[i], y_grid[i+1]
#hat basis function 
        b_left = (x_right - x) / (x_right - x_left)
        b_right = (x - x_left) / (x_right - x_left)

        y_interpolated = y_left * b_left + y_right * b_right # Σf(x_i) * b_i(x) the formula 
        splinevalues.append(y_interpolated)

    return np.array(splinevalues)

plt.figure(figsize=(12, 8))
plt.plot(x_eval, y_true_values, label=r'$f(x)=x^{1/4}$', linewidth=2)

ERROR CONVERGENCE RATE LINEAR SPLINES 

In [ ]:
errors_list = []
power = range(2, 11)
n_list = [2**k for k in power]
for n in n_list:
    x_grid = np.linspace(a, b, n + 1)
    y_grid = x_grid**(1/4)

    linear_values = LinearSplines(x_grid, y_grid, x_eval)  #our interpolation 
    error = np.max(np.abs(y_true_values - linear_values)) #error formula 

    errors_list.append(error) 

    plt.plot(x_eval, linear_values, label=f'n={n}')

plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear spline interpolation of $f(x)=x^{1/4}$ on [0,1]")
plt.legend()
plt.show()


ERROR CONVERGENCE SPEED for linear splines 

To determine the speed of convergence, you need to plot the reference convergence
slopes Eref,m(n) =(1/n)^ m = h^m,

In [ ]:
errors_array = np.array(errors_list)
reference = errors_array[-1] * (n_list[-1] / np.array(n_list))**2 #TAKING THE LAST POINT  

plt.figure(figsize=(8, 6))
plt.loglog(n_list, errors_array, label='Interpolation error')
plt.loglog(n_list, reference, label=r'Reference slope $n^{-1/4}$')
plt.xlabel('n')
plt.ylabel('Maximum error')
plt.title('Log-log plot of interpolation error')
plt.legend()
plt.show()

print("Convegrence speed of the errors is algebraic with m= 2")

PIECEWIESE CONSTANT SPLINE APPROXIMATION 

In [ ]:
def piecewise_constant_approximation(set_ordered_gridpoints, set_evaluationpoints ):
    x_grid = set_ordered_gridpoints #nodes
    x_eval = set_evaluationpoints #evaluation points 
    approximation=[]
    
     # x_grid looks like [x0, x1, ... x{n-1}]
    for x in x_eval: #go through every values of x we want to interpolate
        if x <= x_grid[0]:  # if x is smaller or equal to the 1st grid point
            i = 0           #interval index is 0, we use the 1st interval [x_grid[0], x_grid[1]]
        elif x >= x_grid[-1]:  # if x is greater or equal to the last grid point
            i = len(x_grid) - 2 #since indeces are from 0 to n-1 the last interval is [x_{n-2}, x_{n-1}] 
        else: #now points inside of the domain 
            i = 0 #will be overwritten 
            for j in range(len(x_grid) - 1): 
                if x_grid[j] <= x <= x_grid[j + 1]: #check if x is inside
                    i = j #gives the index for the interval 
                    break
        x_left, x_right= x_grid[i], x_grid[i+1] #interval where x lies 
        y_interpolated= (4/3)*((x_right**(3/4) - x_left**(3/4))/(x_right-x_left)) #OVERALL CELL AVERAGE (comes from integral over the interval)
        approximation.append(y_interpolated)
        
    return approximation  

#PLOT 
f_opt= piecewise_constant_approximation(x_grid, x_eval)


plt.figure(figsize=(12, 8))
plt.plot(x_eval, y_true_values, label=r"$f(x)=x^{-1/4}$")
plt.step(x_eval, f_opt, label=r"$f_{\mathrm{opt}}$")

plt.xlabel("x grid ")
plt.ylabel("y grid")
plt.title(" f_optimal")
plt.legend()
plt.show()

Identity matrix 

In [ ]:
import numpy as np 
np.eye(n)  


Matrix * Transpose matrix

In [ ]:
np.outer(A, A)